In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, LSTM, Embedding, Dense,
    Dot, Activation, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
EMBEDDING_DIM = 32
LATENT_DIM = 64
EPOCHS = 300

In [ ]:
eng_sentences = [
    "i eat apples",
    "i like dogs"
]

fra_sentences = [
    "je mange des pommes",
    "j aime les chiens"
]

# add special tokens
fra_sentences = [f"<start> {s} <end>" for s in fra_sentences]


In [ ]:
enc_tokenizer = Tokenizer()
enc_tokenizer.fit_on_texts(eng_sentences)

dec_tokenizer = Tokenizer(filters="")
dec_tokenizer.fit_on_texts(fra_sentences)


encoder_input = enc_tokenizer.texts_to_sequences(eng_sentences)
decoder_input = dec_tokenizer.texts_to_sequences(fra_sentences)

decoder_target = [seq[1:] for seq in decoder_input]
decoder_input  = [seq[:-1] for seq in decoder_input]


max_enc_len = max(len(s) for s in encoder_input)
max_dec_len = max(len(s) for s in decoder_input)

encoder_input = pad_sequences(encoder_input, maxlen=max_enc_len, padding="post")
decoder_input = pad_sequences(decoder_input, maxlen=max_dec_len, padding="post")
decoder_target = pad_sequences(decoder_target, maxlen=max_dec_len, padding="post")


In [ ]:
encoder_inputs = Input(shape=(None,), name="encoder_inputs")

encoder_embedding = Embedding(
    len(enc_tokenizer.word_index) + 1,
    EMBEDDING_DIM,
    name="encoder_embedding"
)

enc_embed = encoder_embedding(encoder_inputs)

encoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(enc_embed)
encoder_states = [state_h, state_c]


In [ ]:
decoder_inputs = Input(shape=(None,), name="decoder_inputs")

decoder_embedding = Embedding(
    len(dec_tokenizer.word_index) + 1,
    EMBEDDING_DIM,
    name="decoder_embedding"
)

dec_embed = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    dec_embed,
    initial_state=encoder_states
)


In [ ]:
# similarity scores
attention_scores = Dot(axes=[2, 2], name="attention_scores")(
    [decoder_outputs, encoder_outputs]
)

# attention weights
attention_weights = Activation("softmax", name="attention_weights")(
    attention_scores
)

# context vector
context = Dot(axes=[2, 1], name="context_vector")(
    [attention_weights, encoder_outputs]
)

# combine context + decoder output
decoder_combined = Concatenate(axis=-1, name="decoder_context")(
    [context, decoder_outputs]
)


decoder_dense = Dense(
    len(dec_tokenizer.word_index) + 1,
    activation="softmax",
    name="output_dense"
)

decoder_outputs = decoder_dense(decoder_combined)


In [ ]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)

model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=EPOCHS,
    verbose=0
)


# Encoder Inference

encoder_model = Model(
    encoder_inputs,
    [encoder_outputs] + encoder_states
)

# Decoder Inference

decoder_state_input_h = Input(shape=(LATENT_DIM,))
decoder_state_input_c = Input(shape=(LATENT_DIM,))
encoder_outputs_input = Input(shape=(None, LATENT_DIM))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_embed_inf = decoder_embedding(decoder_inputs)

decoder_outputs_inf, h_inf, c_inf = decoder_lstm(
    dec_embed_inf,
    initial_state=decoder_states_inputs
)

attention_scores_inf = Dot(axes=[2, 2])(
    [decoder_outputs_inf, encoder_outputs_input]
)
attention_weights_inf = Activation("softmax")(attention_scores_inf)

context_inf = Dot(axes=[2, 1])(
    [attention_weights_inf, encoder_outputs_input]
)

decoder_combined_inf = Concatenate(axis=-1)(
    [context_inf, decoder_outputs_inf]
)

decoder_outputs_inf = decoder_dense(decoder_combined_inf)

decoder_model = Model(
    [decoder_inputs, encoder_outputs_input] + decoder_states_inputs,
    [decoder_outputs_inf, h_inf, c_inf]
)


In [ ]:
def translate(sentence):
    seq = enc_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_enc_len, padding="post")

    encoder_outputs, h, c = encoder_model.predict(seq)

    word = "<start>"
    result = []

    for _ in range(max_dec_len):
        token = dec_tokenizer.texts_to_sequences([[word]])
        token = pad_sequences(token, maxlen=1)

        preds, h, c = decoder_model.predict(
            [token, encoder_outputs, h, c]
        )

        word_id = np.argmax(preds[0, 0])
        word = dec_tokenizer.index_word[word_id]

        if word == "<end>":
            break

        result.append(word)

    return " ".join(result)
